In [ ]:
!pip install dgl -f https://data.dgl.ai/wheels/torch-2.3/cu121/repo.html -q
!pip install category_encoders -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.5/370.5 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 8.7 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import dgl
print("DGL version:", dgl.__version__)

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import dgl.function as fn
import networkx as nx
import category_encoders as ce
import math, time, pickle, joblib, random
import tqdm
from typing import *
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler, Normalizer
from sklearn.model_selection import train_test_split

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

DGL backend not selected or invalid.  Assuming PyTorch for now.


Setting the default backend to "pytorch". You can change it in the ~/.dgl/config.json file or export the DGLBACKEND environment variable.  Valid options are: pytorch, mxnet, tensorflow (all lowercase)
DGL version: 2.5.0+cu121
Using device: cuda


In [ ]:
file_name = '/content/drive/MyDrive/NF-CSE-CIC-IDS2018-v2.csv'

# Read only a chunk instead of full file
data = pd.read_csv(file_name, nrows=500000)  # only 500k rows

print(f"Dataset size: {len(data):,} rows")
print(f"Benign : {sum(data['Label']==0):,}")
print(f"Attacks: {sum(data['Label']==1):,}")

Dataset size: 500,000 rows
Benign : 439,972
Attacks: 60,028


In [ ]:
data.rename(columns=lambda x: x.strip(), inplace=True)
data['IPV4_SRC_ADDR'] = data['IPV4_SRC_ADDR'].apply(str)
data['IPV4_DST_ADDR'] = data['IPV4_DST_ADDR'].apply(str)
data['L4_SRC_PORT']   = data['L4_SRC_PORT'].apply(str)
data['L4_DST_PORT']   = data['L4_DST_PORT'].apply(str)
data.drop(columns=['L4_SRC_PORT', 'L4_DST_PORT'], inplace=True)

print("Attack types:", data.Attack.unique())

Attack types: ['SSH-Bruteforce' 'Benign' 'DDoS attacks-LOIC-HTTP' 'DDOS attack-HOIC'
 'DoS attacks-Slowloris' 'DoS attacks-Hulk' 'FTP-BruteForce'
 'Infilteration' 'Bot' 'DoS attacks-GoldenEye' 'Brute Force -Web'
 'DoS attacks-SlowHTTPTest' 'SQL Injection' 'DDOS attack-LOIC-UDP'
 'Brute Force -XSS']


In [ ]:
X = data.drop(columns=['Attack', 'Label'])
y = data[['Attack', 'Label']]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=13, stratify=y)

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")

Train: 350,000 | Test: 150,000


In [ ]:
X = data.drop(columns=['Attack', 'Label'])
y = data[['Attack', 'Label']]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=13, stratify=y)

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")

Train: 350,000 | Test: 150,000


In [ ]:
encoder = ce.TargetEncoder(cols=[
    'TCP_FLAGS','L7_PROTO','PROTOCOL',
    'CLIENT_TCP_FLAGS','SERVER_TCP_FLAGS','ICMP_TYPE',
    'ICMP_IPV4_TYPE','DNS_QUERY_ID','DNS_QUERY_TYPE',
    'FTP_COMMAND_RET_CODE'
])
encoder.fit(X_train, y_train.Label)
X_train = encoder.transform(X_train)
X_test  = encoder.transform(X_test)
print("Encoding done.")

Encoding done.


In [ ]:
X_train.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)
X_train.fillna(0, inplace=True)
X_test.fillna(0, inplace=True)
print("Done.")

Done.


In [ ]:
scaler = Normalizer()
cols_to_norm = list(set(list(X_train.iloc[:, 2:].columns)))
scaler.fit(X_train[cols_to_norm])

X_train[cols_to_norm] = scaler.transform(X_train[cols_to_norm])
X_train['h'] = X_train.iloc[:, 2:].values.tolist()

X_test[cols_to_norm] = scaler.transform(X_test[cols_to_norm])
X_test['h'] = X_test.iloc[:, 2:].values.tolist()

train = pd.concat([X_train, y_train], axis=1)
test  = pd.concat([X_test,  y_test],  axis=1)
print("Feature vector length:", len(train['h'].iloc[0]))

Feature vector length: 39


In [ ]:
lab_enc = preprocessing.LabelEncoder()
lab_enc.fit(data['Attack'])
train['Attack'] = lab_enc.transform(train['Attack'])
test['Attack']  = lab_enc.transform(test['Attack'])
joblib.dump(lab_enc, 'gnn_label_encoder.pkl', compress=9)
print("Classes:", list(lab_enc.classes_))

Classes: ['Benign', 'Bot', 'Brute Force -Web', 'Brute Force -XSS', 'DDOS attack-HOIC', 'DDOS attack-LOIC-UDP', 'DDoS attacks-LOIC-HTTP', 'DoS attacks-GoldenEye', 'DoS attacks-Hulk', 'DoS attacks-SlowHTTPTest', 'DoS attacks-Slowloris', 'FTP-BruteForce', 'Infilteration', 'SQL Injection', 'SSH-Bruteforce']


In [ ]:
train_g = nx.from_pandas_edgelist(
    train, 'IPV4_SRC_ADDR', 'IPV4_DST_ADDR',
    ['h', 'Label', 'Attack'], create_using=nx.MultiGraph())
train_g = train_g.to_directed()
train_g = dgl.from_networkx(train_g, edge_attrs=['h', 'Attack', 'Label'])
train_g.ndata['h'] = torch.ones([train_g.number_of_nodes(), train_g.edata['h'].shape[1]])

test_g = nx.from_pandas_edgelist(
    test, 'IPV4_SRC_ADDR', 'IPV4_DST_ADDR',
    ['h', 'Label', 'Attack'], create_using=nx.MultiGraph())
test_g = test_g.to_directed()
test_g = dgl.from_networkx(test_g, edge_attrs=['h', 'Attack', 'Label'])
test_g.ndata['h'] = torch.ones([test_g.number_of_nodes(), test_g.edata['h'].shape[1]])

print(f"Train graph — Nodes: {train_g.number_of_nodes():,} | Edges: {train_g.number_of_edges():,}")
print(f"Test  graph — Nodes: {test_g.number_of_nodes():,}  | Edges: {test_g.number_of_edges():,}")

Train graph — Nodes: 28,070 | Edges: 699,060
Test  graph — Nodes: 16,056  | Edges: 299,578


In [ ]:
dgl.save_graphs('train.graph', [train_g])
dgl.save_graphs('test.graph',  [test_g])
print("Graphs saved.")

Graphs saved.


In [ ]:
class SAGELayer(nn.Module):
    def __init__(self, ndim_in, edims, ndim_out, activation):
        super(SAGELayer, self).__init__()
        self.W_apply = nn.Linear(ndim_in + edims, ndim_out)
        self.activation = F.relu
        self.W_edge = nn.Linear(128 * 2, 256)
        self.reset_parameters()

    def reset_parameters(self):
        gain = nn.init.calculate_gain('relu')
        nn.init.xavier_uniform_(self.W_apply.weight, gain=gain)

    def message_func(self, edges):
        return {'m': edges.data['h']}

    def forward(self, g_dgl, nfeats, efeats):
        with g_dgl.local_scope():
            g = g_dgl
            g.ndata['h'] = nfeats
            g.edata['h'] = efeats
            g.update_all(self.message_func, fn.mean('m', 'h_neigh'))
            g.ndata['h'] = F.relu(self.W_apply(torch.cat([g.ndata['h'], g.ndata['h_neigh']], 2)))
            u, v = g.edges()
            edge = self.W_edge(torch.cat((g.srcdata['h'][u], g.dstdata['h'][v]), 2))
            return g.ndata['h'], edge


class SAGE(nn.Module):
    def __init__(self, ndim_in, ndim_out, edim, activation):
        super(SAGE, self).__init__()
        self.layers = nn.ModuleList()
        self.layers.append(SAGELayer(ndim_in, edim, 128, F.relu))

    def forward(self, g, nfeats, efeats, corrupt=False):
        if corrupt:
            e_perm = torch.randperm(g.number_of_edges())
            efeats = efeats[e_perm]
        for layer in self.layers:
            nfeats, e_feats = layer(g, nfeats, efeats)
        return nfeats.sum(1), e_feats.sum(1)


class Discriminator(nn.Module):
    def __init__(self, n_hidden):
        super(Discriminator, self).__init__()
        self.weight = nn.Parameter(torch.Tensor(n_hidden, n_hidden))
        self.reset_parameters()

    def uniform(self, size, tensor):
        bound = 1.0 / math.sqrt(size)
        if tensor is not None:
            tensor.data.uniform_(-bound, bound)

    def reset_parameters(self):
        self.uniform(self.weight.size(0), self.weight)

    def forward(self, features, summary):
        return torch.matmul(features, torch.matmul(self.weight, summary))


class DGI(nn.Module):
    def __init__(self, ndim_in, ndim_out, edim, activation):
        super(DGI, self).__init__()
        self.encoder = SAGE(ndim_in, ndim_out, edim, F.relu)
        self.discriminator = Discriminator(256)
        self.loss = nn.BCEWithLogitsLoss()

    def forward(self, g, n_features, e_features):
        positive = self.encoder(g, n_features, e_features, corrupt=False)
        negative = self.encoder(g, n_features, e_features, corrupt=True)
        positive = positive[1]
        negative = negative[1]
        summary  = torch.sigmoid(positive.mean(dim=0))
        positive = self.discriminator(positive, summary)
        negative = self.discriminator(negative, summary)
        l1 = self.loss(positive, torch.ones_like(positive))
        l2 = self.loss(negative, torch.zeros_like(negative))
        return l1 + l2

print("Model classes defined.")

Model classes defined.


In [ ]:
ndim_in  = train_g.ndata['h'].shape[1]
ndim_out = 128
edim     = train_g.edata['h'].shape[1]
epochs   = 2000
learning_rate = 1e-3

dgi = DGI(ndim_in, ndim_out, edim, F.relu).to(device)
dgi_optimizer = torch.optim.Adam(dgi.parameters(), lr=learning_rate, weight_decay=0.)

# Reshape to 3D
train_g.ndata['h'] = torch.reshape(train_g.ndata['h'],
    (train_g.ndata['h'].shape[0], 1, train_g.ndata['h'].shape[1]))
train_g.edata['h'] = torch.reshape(train_g.edata['h'],
    (train_g.edata['h'].shape[0], 1, train_g.edata['h'].shape[1]))

# Move graph to GPU
train_g = train_g.to(device)

node_features = train_g.ndata['h']
edge_features = train_g.edata['h']

print(f"Node feature shape: {node_features.shape}")
print(f"Edge feature shape: {edge_features.shape}")
print(f"Device: {device}")

Node feature shape: torch.Size([28070, 1, 39])
Edge feature shape: torch.Size([699060, 1, 39])
Device: cuda


In [ ]:
cnt_wait = 0
best = 1e9
best_t = 0
dur = []

print("Starting DGI training...")
for epoch in range(epochs):
    dgi.train()
    if epoch >= 3:
        t0 = time.time()

    dgi_optimizer.zero_grad()
    loss = dgi(train_g, node_features, edge_features)
    loss.backward()
    dgi_optimizer.step()

    if loss < best:
        best = loss
        best_t = epoch
        cnt_wait = 0
        torch.save(dgi.state_dict(), 'best_dgi.pkl')
    else:
        cnt_wait += 1

    if epoch >= 3:
        dur.append(time.time() - t0)

    if epoch % 50 == 0:
        print("Epoch {:05d} | Loss {:.4f}".format(epoch, loss.item()))

print("Training done. Best epoch:", best_t)

Starting DGI training...
Epoch 00000 | Loss 0.1583
Epoch 00050 | Loss 0.1525
Epoch 00100 | Loss 0.1526
Epoch 00150 | Loss 0.1509
Epoch 00200 | Loss 0.1423
Epoch 00250 | Loss 0.1456
Epoch 00300 | Loss 0.1342
Epoch 00350 | Loss 0.1322
Epoch 00400 | Loss 0.1362
Epoch 00450 | Loss 0.1497
Epoch 00500 | Loss 0.1295
Epoch 00550 | Loss 0.1317
Epoch 00600 | Loss 0.1197
Epoch 00650 | Loss 0.1075
Epoch 00700 | Loss 0.0993
Epoch 00750 | Loss 0.0961
Epoch 00800 | Loss 0.1166
Epoch 00850 | Loss 0.0915
Epoch 00900 | Loss 0.1174
Epoch 00950 | Loss 0.0736
Epoch 01000 | Loss 0.0734
Epoch 01050 | Loss 5.5858
Epoch 01100 | Loss 1.3175
Epoch 01150 | Loss 1.2769
Epoch 01200 | Loss 0.5650
Epoch 01250 | Loss 0.4434
Epoch 01300 | Loss 0.4126
Epoch 01350 | Loss 0.3872
Epoch 01400 | Loss 0.3709
Epoch 01450 | Loss 0.3536
Epoch 01500 | Loss 0.3402
Epoch 01550 | Loss 0.3288
Epoch 01600 | Loss 0.3110
Epoch 01650 | Loss 0.2938
Epoch 01700 | Loss 0.2784
Epoch 01750 | Loss 0.2572
Epoch 01800 | Loss 0.2450
Epoch 01850 |

In [ ]:
dgi.load_state_dict(torch.load('best_dgi.pkl'))
dgi.eval()

with torch.no_grad():
    training_emb = dgi.encoder(train_g, train_g.ndata['h'], train_g.edata['h'])[1]
training_emb = training_emb.detach().cpu().numpy()

# Reshape and move test graph
test_g.ndata['h'] = torch.reshape(test_g.ndata['h'],
    (test_g.ndata['h'].shape[0], 1, test_g.ndata['h'].shape[1]))
test_g.edata['h'] = torch.reshape(test_g.edata['h'],
    (test_g.edata['h'].shape[0], 1, test_g.edata['h'].shape[1]))
test_g = test_g.to(device)

with torch.no_grad():
    testing_emb = dgi.encoder(test_g, test_g.ndata['h'], test_g.edata['h'])[1]
testing_emb = testing_emb.detach().cpu().numpy()

print(f"Train embeddings: {training_emb.shape}")
print(f"Test  embeddings: {testing_emb.shape}")

Train embeddings: (699060, 256)
Test  embeddings: (299578, 256)


In [ ]:
# Save in chunks to avoid RAM crash
import gc

# Train embeddings
df_train = pd.DataFrame(training_emb)
df_train['Attack'] = lab_enc.inverse_transform(
    train_g.edata['Attack'].detach().cpu().numpy().astype(int))
df_train['Label'] = train_g.edata['Label'].detach().cpu().numpy()
df_train.columns = df_train.columns.astype(str)
df_train.to_csv('train_embedded.csv', index=False)
print("Train saved.")

# Free RAM before test
del df_train
gc.collect()

# Test embeddings
df_test = pd.DataFrame(testing_emb)
df_test['Attack'] = lab_enc.inverse_transform(
    test_g.edata['Attack'].detach().cpu().numpy().astype(int))
df_test['Label'] = test_g.edata['Label'].detach().cpu().numpy()
df_test.columns = df_test.columns.astype(str)
df_test.to_csv('test_embedded.csv', index=False)
print("Test saved.")

del df_test
gc.collect()
print("Done.")

Train saved.
Test saved.
Done.


In [ ]:
import shutil, os

save_dir = '/content/drive/MyDrive/xcba_outputs/'
os.makedirs(save_dir, exist_ok=True)

for f in ['train_embedded.csv', 'test_embedded.csv',
          'train.graph', 'test.graph',
          'gnn_label_encoder.pkl', 'best_dgi.pkl']:
    shutil.copy(f, save_dir)
    print(f"Saved {f}")

print("All outputs saved to Google Drive.")

Saved train_embedded.csv
Saved test_embedded.csv
Saved train.graph
Saved test.graph
Saved gnn_label_encoder.pkl
Saved best_dgi.pkl
All outputs saved to Google Drive.
